# Inference Performance Table
Reads  files from each PDE's  directory,
filters to the best WFM wavelet per PDE, computes derived metrics,
and outputs a publication-ready LaTeX table.

In [1]:
# ── Cell 1 — Imports ──────────────────────────────────────────────────────────
import yaml
import numpy as np
import pandas as pd
from pathlib import Path

In [ ]:
# ── Cell 2 — Configuration ────────────────────────────────────────────────────
PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())  # repo root

PDES = [
    "turbulent_radiative_layer_2D",
    "rayleigh_benard",
    "active_matter",
]

# Glob prefixes used to discover YAML files inside each performance/ directory
MODEL_PATTERNS = [
    "FM_data_space",
    "WaveletFlow_1scale",
    "WaveletFlow_3scales",
]

NFE = 50  # NFE used during inference (same for all models)

# Best wavelet per (PDE, scale) — used to filter df before display
PDE_BEST_CONFIG = {
    "turbulent_radiative_layer_2D": {
        "WaveletFlow_1scale":  "WaveletFlow_1scale_haar",
        "WaveletFlow_3scales": "WaveletFlow_3scales_haar",
    },
    "rayleigh_benard": {
        "WaveletFlow_1scale":  "WaveletFlow_1scale_db4",
        "WaveletFlow_3scales": "WaveletFlow_3scales_db4",
    },
    "active_matter": {
        "WaveletFlow_1scale":  "WaveletFlow_1scale_haar",
        "WaveletFlow_3scales": "WaveletFlow_3scales_db2",
    },
}

# Display labels (order defines table row order)
MODEL_LABELS = {
    "FM_data_space":      r"$\mathrm{FM}_{\mathrm{pixel}}$",
    "WaveletFlow_1scale": "WFM 1-scale",
    "WaveletFlow_3scales": "WFM 3-scale",
}
MODEL_ORDER = list(MODEL_LABELS.values())

PDE_LABELS = {
    "turbulent_radiative_layer_2D": "Turb. Rad. Layer 2D",
    "rayleigh_benard":              "Rayleigh-Bénard",
    "active_matter":                "Active Matter",
}
PDE_ORDER = list(PDE_LABELS.values())

# (raw_col, display_name, format_spec, show_for_FM_baseline)
METRICS = [
    ("wall_clock_s",          "Wall Clock (s)",      ".1f",  True),
    ("sec_per_rollout",       "sec / step",          ".4f",  True),
    ("gpu_peak_mem_GiB",      "GPU Mem (GiB)",       ".2f",  True),
    ("throughput_Mframes_s",  "Throughput (Mfr/s)",  ".2f",  True),
    ("ratio_wall_clock_s",    "Speedup ×",           ".2f",  False),
    ("ratio_gpu_peak_mem_GiB","Mem ratio ×",         ".2f",  False),
]
METRIC_COLS = [label for _, label, _, _ in METRICS]
FMT_MAP     = {label: fmt for _, label, fmt, _ in METRICS}

# Ratio metrics: "lower" → FM/WFM (speedup), "higher" → WFM/FM (throughput)
RATIO_METRICS = [
    ("wall_clock_s",         "lower"),
    ("sec_per_rollout",      "lower"),
    ("gpu_peak_mem_GiB",     "lower"),
    ("throughput_Mframes_s", "higher"),
]

In [3]:
# ── Cell 3 — YAML helpers ─────────────────────────────────────────────────────
def avg_gpu_mem(block) -> float | None:
    """Mean peak GPU memory across all ranks (GiB)."""
    if block is None:
        return None
    if isinstance(block, (int, float)):
        return float(block)
    if isinstance(block, dict):
        vals = [float(v) for v in block.values() if v is not None]
        return float(np.mean(vals)) if vals else None
    return None


def resolve_model_name(filename: str, pde: str) -> str:
    """Strip _{pde}_perf.yaml / _{pde}_performance.yaml to get the model key."""
    for suffix in (f"_{pde}_perf.yaml", f"_{pde}_performance.yaml"):
        if filename.endswith(suffix):
            return filename[: -len(suffix)]
    return Path(filename).stem


def load_yaml(path: Path) -> dict:
    with open(path) as f:
        raw = yaml.safe_load(f)
    wc = raw.get("wall_clock_s")
    ts = raw.get("total_steps_global")
    return {
        "wall_clock_s":        wc,
        "total_steps_global":  ts,
        "gpu_peak_mem_GiB":    avg_gpu_mem(raw.get("gpu_peak_mem_GiB")),
        "throughput_steps_s":  raw.get("throughput_steps_s"),
        "throughput_frames_s": raw.get("throughput_frames_s"),
        "sec_per_rollout":     round(wc / ts, 6) if (wc and ts) else None,
    }

In [4]:
# ── Cell 4 — Discover and load all YAML files ─────────────────────────────────
records = []
for pde in PDES:
    perf_dir = PROJECT_ROOT / "experiments" / pde / "performance"
    if not perf_dir.exists():
        print(f"[MISS] {perf_dir}")
        continue

    found: dict[str, Path] = {}
    for pattern in MODEL_PATTERNS:
        for p in (
            sorted(perf_dir.glob(f"{pattern}*_perf.yaml"))
            + sorted(perf_dir.glob(f"{pattern}*_performance.yaml"))
        ):
            name = resolve_model_name(p.name, pde)
            found.setdefault(name, p)   # first match wins

    if not found:
        print(f"[MISS] no matching YAMLs in {perf_dir}")
        continue

    for model_name, yaml_path in sorted(found.items()):
        print(f"  ✓  {yaml_path.name}")
        try:
            records.append({"pde": pde, "model": model_name, **load_yaml(yaml_path)})
        except Exception as e:
            print(f"    [ERROR] {yaml_path}: {e}")

df = pd.DataFrame(records)
df["throughput_Mframes_s"] = df["throughput_frames_s"] / 1e6
print(f"\nLoaded {len(df)} rows")
df

  ✓  FM_data_space_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_1scale_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_1scale_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_1scale_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_1scale_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_db2_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_db4_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_db6_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  WaveletFlow_3scales_haar_v4.0_nsampling50_nens8_turbulent_radiative_layer_2D_perf.yaml
  ✓  FM_data_space_v4.0_nsampling50_nens8_rayleigh_benard_perf.yaml
  ✓  WaveletFlow_1scale_db2_v4.0_nsampling50_nens8_rayleigh_benard_perf.yaml
  ✓  WaveletFlow_1scale_db4_v4.0_nsampling50_nen

,pde,model,wall_clock_s,total_steps_global,gpu_peak_mem_GiB,throughput_steps_s,throughput_frames_s,sec_per_rollout,throughput_Mframes_s
0,turbulent_radiative_layer_2D,FM_data_space_v4.0_nsampling50_nens8,247.75,6984,0.59000,28.2,1385580.0,0.035474,1.385580
1,turbulent_radiative_layer_2D,WaveletFlow_1scale_db2_v4.0_nsampling50_nens8,199.04,6984,0.59200,35.1,1724697.7,0.028499,1.724698
2,turbulent_radiative_layer_2D,WaveletFlow_1scale_db4_v4.0_nsampling50_nens8,208.10,6984,0.59100,33.6,1649548.1,0.029797,1.649548
3,turbulent_radiative_layer_2D,WaveletFlow_1scale_db6_v4.0_nsampling50_nens8,204.52,6984,0.59200,34.1,1678469.7,0.029284,1.678470
4,turbulent_radiative_layer_2D,WaveletFlow_1scale_haar_v4.0_nsampling50_nens8,198.06,6984,0.59100,35.3,1733232.8,0.028359,1.733233
5,turbulent_radiative_layer_2D,WaveletFlow_3scales_db2_v4.0_nsampling50_nens8,224.02,6984,0.66300,31.2,1532380.0,0.032076,1.532380
6,turbulent_radiative_layer_2D,WaveletFlow_3scales_db4_v4.0_nsampling50_nens8,228.23,6984,0.66300,30.6,1504072.0,0.032679,1.504072
7,turbulent_radiative_layer_2D,WaveletFlow_3scales_db6_v4.0_nsampling50_nens8,221.05,6984,0.66300,31.6,1552952.8,0.031651,1.552953
8,turbulent_radiative_layer_2D,WaveletFlow_3scales_haar_v4.0_nsampling50_nens8,218.27,6984,0.66300,32.0,1572733.1,0.031253,1.572733
9,rayleigh_benard,FM_data_space_v4.0_nsampling50_nens8,633.85,14112,1.44100,22.3,1459093.4,0.044916,1.459093


In [5]:
# ── Cell 5 — Classify, filter to best configs, compute FM-relative ratios ──────
WFM_KEYWORDS = ["WaveletFlow", "WFM", "Wavelet"]

def is_wfm(name: str) -> bool:
    return any(k.lower() in name.lower() for k in WFM_KEYWORDS)

def is_best_wfm(row) -> bool:
    """
    Keep FM baselines always.
    Keep a WFM row only when it matches the PDE-optimal wavelet config.
    """
    if not is_wfm(row["model"]):
        return True
    pde_cfg = PDE_BEST_CONFIG.get(row["pde"], {})
    return any(
        scale_key in row["model"] and row["model"].startswith(best_prefix)
        for scale_key, best_prefix in pde_cfg.items()
    )

df["model_type"] = df["model"].apply(lambda m: "WFM" if is_wfm(m) else "FM")

# Filter to best-per-PDE configs before computing ratios
df_best = df[df.apply(is_best_wfm, axis=1)].copy()

# Initialise ratio columns
for metric, _ in RATIO_METRICS:
    df_best[f"ratio_{metric}"] = np.nan

# Compute ratios relative to FM baseline within each PDE
for pde, grp in df_best.groupby("pde"):
    fm_rows = grp[grp["model_type"] == "FM"]
    if fm_rows.empty:
        print(f"[WARN] no FM baseline for {pde}")
        continue
    fm_row = fm_rows.iloc[0]
    for idx in grp[grp["model_type"] == "WFM"].index:
        for metric, direction in RATIO_METRICS:
            fv = fm_row.get(metric)
            wv = df_best.at[idx, metric]
            if pd.notna(fv) and pd.notna(wv) and fv != 0:
                df_best.at[idx, f"ratio_{metric}"] = (
                    wv / fv if direction == "higher" else fv / wv
                )

df_best[[
    "pde", "model", "model_type",
    "wall_clock_s", "sec_per_rollout", "gpu_peak_mem_GiB", "throughput_Mframes_s",
    "ratio_wall_clock_s", "ratio_gpu_peak_mem_GiB",
]]

,pde,model,model_type,wall_clock_s,sec_per_rollout,gpu_peak_mem_GiB,throughput_Mframes_s,ratio_wall_clock_s,ratio_gpu_peak_mem_GiB
0,turbulent_radiative_layer_2D,FM_data_space_v4.0_nsampling50_nens8,FM,247.75,0.035474,0.590,1.385580,NaN,NaN
4,turbulent_radiative_layer_2D,WaveletFlow_1scale_haar_v4.0_nsampling50_nens8,WFM,198.06,0.028359,0.591,1.733233,1.250884,0.998308
8,turbulent_radiative_layer_2D,WaveletFlow_3scales_haar_v4.0_nsampling50_nens8,WFM,218.27,0.031253,0.663,1.572733,1.135062,0.889894
9,rayleigh_benard,FM_data_space_v4.0_nsampling50_nens8,FM,633.85,0.044916,1.441,1.459093,NaN,NaN
11,rayleigh_benard,WaveletFlow_1scale_db4_v4.0_nsampling50_nens8,WFM,418.13,0.029629,1.444,2.211869,1.515916,0.997922
15,rayleigh_benard,WaveletFlow_3scales_db4_v4.0_nsampling50_nens8,WFM,457.89,0.032447,1.515,2.019778,1.384284,0.951155
18,active_matter,FM_data_space_v4.0_nsampling50_nens8,FM,310.77,0.045863,1.629,1.428956,NaN,NaN
22,active_matter,WaveletFlow_1scale_haar_v4.0_nsampling50_nens8,WFM,213.28,0.031476,1.636,2.082143,1.457099,0.995721
23,active_matter,WaveletFlow_3scales_db2_v4.0_nsampling50_nens8,WFM,247.36,0.036505,1.713,1.795235,1.256347,0.950963


In [6]:
# ── Cell 6 — Annotate and build display pivot ─────────────────────────────────
def shorten_model(name: str) -> str:
    """WaveletFlow_1scale_haar_v4.0 → 'WFM 1-scale' (prefix match)."""
    for key, label in MODEL_LABELS.items():
        if name.startswith(key):
            return label
    return name

def extract_wavelet(name: str) -> str:
    for w in ["haar", "db2", "db4", "db6"]:
        if w in name:
            return w
    return ""

df2 = df_best.copy()
df2["model_short"] = df2["model"].apply(shorten_model)
df2["pde_label"]   = df2["pde"].map(PDE_LABELS)
df2["wavelet"]     = df2["model"].apply(extract_wavelet)

def fmt_cell(val, metric_name: str) -> str:
    if pd.isna(val):
        return "—"
    s = f"{val:{FMT_MAP.get(metric_name, '.3f')}}"
    return s + "×" if "×" in metric_name else s

rows, index = [], []
for pde_label in PDE_ORDER:
    for model in MODEL_ORDER:
        is_fm = "FM" in model and "WFM" not in model
        cell  = df2[(df2["pde_label"] == pde_label) & (df2["model_short"] == model)]
        row   = {}
        for raw_col, label, _, show_for_fm in METRICS:
            if cell.empty or raw_col not in cell.columns:
                row[label] = np.nan
            elif is_fm and not show_for_fm:
                row[label] = np.nan
            else:
                row[label] = cell[raw_col].values[0]
        rows.append(row)
        index.append((pde_label, model))

pivot = pd.DataFrame(
    rows, columns=METRIC_COLS,
    index=pd.MultiIndex.from_tuples(index, names=["PDE", "Model"]),
)

pivot_fmt = pivot.copy().astype(object)
for metric in METRIC_COLS:
    pivot_fmt[metric] = pivot[metric].apply(lambda v: fmt_cell(v, metric))

pivot_fmt

Wall Clock (s) sec / step  \
PDE                 Model                                                      
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$          247.8     0.0355   
                    WFM 1-scale                             198.1     0.0284   
                    WFM 3-scale                             218.3     0.0313   
Rayleigh-Bénard     $\mathrm{FM}_{\mathrm{pixel}}$          633.9     0.0449   
                    WFM 1-scale                             418.1     0.0296   
                    WFM 3-scale                             457.9     0.0324   
Active Matter       $\mathrm{FM}_{\mathrm{pixel}}$          310.8     0.0459   
                    WFM 1-scale                             213.3     0.0315   
                    WFM 3-scale                             247.4     0.0365   

                                                   GPU Mem (GiB)  \
PDE                 Model                                          
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$          0.59   
                    WFM 1-scale                             0.59   
                    WFM 3-scale                             0.66   
Rayleigh-Bénard     $\mathrm{FM}_{\mathrm{pixel}}$          1.44   
                    WFM 1-scale                             1.44   
                    WFM 3-scale                             1.51   
Active Matter       $\mathrm{FM}_{\mathrm{pixel}}$          1.63   
                    WFM 1-scale                             1.64   
                    WFM 3-scale                             1.71   

                                                   Throughput (Mfr/s)  \
PDE                 Model                                               
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$               1.39   
                    WFM 1-scale                                  1.73   
                    WFM 3-scale                                  1.57   
Rayleigh-Bénard     $\mathrm{FM}_{\mathrm{pixel}}$               1.46   
                    WFM 1-scale                                  2.21   
                    WFM 3-scale                                  2.02   
Active Matter       $\mathrm{FM}_{\mathrm{pixel}}$               1.43   
                    WFM 1-scale                                  2.08   
                    WFM 3-scale                                  1.80   

                                                   Speedup × Mem ratio ×  
PDE                 Model                                                 
Turb. Rad. Layer 2D $\mathrm{FM}_{\mathrm{pixel}}$         —           —  
                    WFM 1-scale                        1.25×       1.00×  
                    WFM 3-scale                        1.14×       0.89×  
Rayleigh-Bénard     $\mathrm{FM}_{\mathrm{pixel}}$         —           —  
                    WFM 1-scale                        1.52×       1.00×  
                    WFM 3-scale                        1.38×       0.95×  
Active Matter       $\mathrm{FM}_{\mathrm{pixel}}$         —           —  
                    WFM 1-scale                        1.46×       1.00×  
                    WFM 3-scale                        1.26×       0.95×

In [9]:
# ── Cell 7 — LaTeX table ──────────────────────────────────────────────────────
PDE_LATEX = {
    "Turb. Rad. Layer 2D": "Turbulent Radiative Layer 2D",
    "Rayleigh-Bénard":     "Rayleigh-B\\'{e}nard",
    "Active Matter":       "Active Matter",
}

MODEL_LATEX = {
    r"$\mathrm{FM}_{\mathrm{pixel}}$": r"$\text{FM}_{\text{pixel}}$",
    "WFM 1-scale": "WFM 1-scale",
    "WFM 3-scale": "WFM 3-scale",
}

caption = (
    r"Inference performance across PDEs and models. "
    r"For each WFM variant the optimal wavelet (selected per PDE) is shown in parentheses. "
    r"\textit{Wall Clock}: total inference time (s), max across GPU ranks. "
    r"\textit{sec/step}: wall-clock time per autoregressive step "
    r"(\texttt{wall\_clock\_s}\,/\,\texttt{total\_steps\_global}), resolution-independent. "
    r"\textit{GPU Mem}: peak GPU memory averaged across ranks (GiB). "
    r"\textit{Throughput}: millions of spatial frames per second "
    r"(\texttt{total\_steps}\,$\times H\times W$\,/\,\texttt{wall\_clock\_s}). "
    r"\textit{Speedup}: wall-clock ratio $\text{FM}_{\text{pixel}}$\,/\,WFM; "
    r"${>}1$ means WFM is faster. "
    r"\textit{Mem ratio}: peak GPU memory ratio $\text{FM}_{\text{pixel}}$\,/\,WFM; "
    r"${>}1$ means WFM uses less memory. "
    r"All runs: NFE\,=\,50 ODE steps, 4 GPUs, 8 ensemble members."
)

lines = [
    r"\begin{table}[ht]",
    r"\caption{" + caption + "}",
    r"\label{tab:inference_perf}",
    r"\resizebox{\textwidth}{!}{%",
    r"\begin{tabular}{ll rrrr rr}",
    r"\toprule",
    (
        r"\textbf{PDE} & \textbf{Model}"
        r" & \textbf{Wall Clock (s)}"
        r" & \textbf{sec / step}"
        r" & \textbf{GPU Mem (GiB)}"
        r" & \textbf{Throughput (Mfr/s)}"
        r" & \textbf{Speedup $\times$}"
        r" & \textbf{Mem ratio $\times$} \\"
    ),
]

for pde_label in PDE_ORDER:
    lines.append(r"\midrule")
    pde_tex = PDE_LATEX.get(pde_label, pde_label)
    lines.append(rf"\multirow{{{len(MODEL_ORDER)}}}{{*}}{{{pde_tex}}}")

    for model in MODEL_ORDER:
        model_tex = MODEL_LATEX.get(model, model)

        # Append wavelet annotation for WFM rows
        cell = df2[(df2["pde_label"] == pde_label) & (df2["model_short"] == model)]
        if not cell.empty and cell["wavelet"].values[0]:
            wavelet   = cell["wavelet"].values[0]
            model_tex = model_tex + rf" {{\scriptsize(\textit{{{wavelet}}})}}"

        row_data = pivot_fmt.loc[(pde_label, model)]
        vals     = " & ".join(
            str(row_data[m]).replace("×", r"$\times$").replace("—", r"\textemdash")
            for m in METRIC_COLS
        )
        lines.append(f"  & {model_tex} & {vals} \\\\")

lines += [
    r"\bottomrule",
    r"\end{tabular}%",
    r"}",
    r"\end{table}",
]

latex = "\n".join(lines)
print(latex)

\begin{table}[ht]
\caption{Inference performance across PDEs and models. For each WFM variant the optimal wavelet (selected per PDE) is shown in parentheses. \textit{Wall Clock}: total inference time (s), max across GPU ranks. \textit{sec/step}: wall-clock time per autoregressive step (\texttt{wall\_clock\_s}\,/\,\texttt{total\_steps\_global}), resolution-independent. \textit{GPU Mem}: peak GPU memory averaged across ranks (GiB). \textit{Throughput}: millions of spatial frames per second (\texttt{total\_steps}\,$\times H\times W$\,/\,\texttt{wall\_clock\_s}). \textit{Speedup}: wall-clock ratio $\text{FM}_{\text{pixel}}$\,/\,WFM; ${>}1$ means WFM is faster. \textit{Mem ratio}: peak GPU memory ratio $\text{FM}_{\text{pixel}}$\,/\,WFM; ${>}1$ means WFM uses less memory. All runs: NFE\,=\,50 ODE steps, 4 GPUs, 8 ensemble members.}
\label{tab:inference_perf}
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll rrrr rr}
\toprule
\textbf{PDE} & \textbf{Model} & \textbf{Wall Clock (s)} & \textbf{sec

In [8]:
# ── Cell 8 — Save to .tex file ────────────────────────────────────────────────
# out = Path("performance_table.tex")
# out.write_text(latex)
# print(f"Saved → {out.resolve()}")

# ALL 

In [11]:
# ── Cell 8 — Full ablation table: ALL WFM variants × all wavelets ─────────────

# ── 8a. Compute FM-relative ratios on the full (unfiltered) df ────────────────
df_full = df.copy()
df_full["model_type"] = df_full["model"].apply(lambda m: "WFM" if is_wfm(m) else "FM")

for metric, _ in RATIO_METRICS:
    df_full[f"ratio_{metric}"] = np.nan

for pde, grp in df_full.groupby("pde"):
    fm_rows = grp[grp["model_type"] == "FM"]
    if fm_rows.empty:
        print(f"[WARN] no FM baseline for {pde}")
        continue
    fm_row = fm_rows.iloc[0]
    for idx in grp[grp["model_type"] == "WFM"].index:
        for metric, direction in RATIO_METRICS:
            fv = fm_row.get(metric)
            wv = df_full.at[idx, metric]
            if pd.notna(fv) and pd.notna(wv) and fv != 0:
                df_full.at[idx, f"ratio_{metric}"] = (
                    wv / fv if direction == "higher" else fv / wv
                )

# ── 8b. Annotate display columns ───────────────────────────────────────────────
SCALE_ORDER = ["WaveletFlow_1scale", "WaveletFlow_3scales"]
WAVELET_ORDER = ["haar", "db2", "db4", "db6"]

def extract_scale_key(name: str) -> str:
    for s in SCALE_ORDER:
        if name.startswith(s):
            return s
    return ""

df_full["scale_key"] = df_full["model"].apply(extract_scale_key)
df_full["wavelet"]   = df_full["model"].apply(extract_wavelet)
df_full["pde_label"] = df_full["pde"].map(PDE_LABELS)

# ── 8c. Build display rows ─────────────────────────────────────────────────────
# Row order per PDE: FM baseline, then 1-scale × wavelets, then 3-scale × wavelets
abl_rows, abl_index = [], []

SCALE_LABELS = {
    "WaveletFlow_1scale":  "WFM 1-scale",
    "WaveletFlow_3scales": "WFM 3-scale",
}

for pde_label in PDE_ORDER:
    # FM baseline
    fm_cell = df_full[
        (df_full["pde_label"] == pde_label) & (df_full["model_type"] == "FM")
    ]
    row = {}
    for raw_col, label, _, show_for_fm in METRICS:
        if fm_cell.empty or raw_col not in fm_cell.columns:
            row[label] = np.nan
        elif not show_for_fm:
            row[label] = np.nan
        else:
            row[label] = fm_cell[raw_col].values[0]
    abl_rows.append(row)
    abl_index.append((pde_label, r"$\text{FM}_{\text{pixel}}$", ""))

    # WFM variants
    for scale_key in SCALE_ORDER:
        for wavelet in WAVELET_ORDER:
            cell = df_full[
                (df_full["pde_label"] == pde_label)
                & (df_full["scale_key"] == scale_key)
                & (df_full["wavelet"] == wavelet)
            ]
            row = {}
            for raw_col, label, _, _ in METRICS:
                if cell.empty or raw_col not in cell.columns:
                    row[label] = np.nan
                else:
                    row[label] = cell[raw_col].values[0]
            abl_rows.append(row)
            abl_index.append((pde_label, SCALE_LABELS[scale_key], wavelet))

abl_pivot = pd.DataFrame(
    abl_rows, columns=METRIC_COLS,
    index=pd.MultiIndex.from_tuples(abl_index, names=["PDE", "Model", "Wavelet"]),
)

abl_pivot_fmt = abl_pivot.copy().astype(object)
for metric in METRIC_COLS:
    abl_pivot_fmt[metric] = abl_pivot[metric].apply(lambda v: fmt_cell(v, metric))

abl_pivot_fmt

# ── 8d. LaTeX ablation table ───────────────────────────────────────────────────
N_WFM_ROWS = len(SCALE_ORDER) * len(WAVELET_ORDER)        # 8
N_BLOCK    = 1 + N_WFM_ROWS                               # FM + 8 WFM = 9 rows per PDE

caption_abl = (
    r"Full inference-performance ablation across all WFM variants (scale $\times$ wavelet) "
    r"relative to $\text{FM}_{\text{pixel}}$. "
    r"Columns are defined as in Table~\ref{tab:inference_perf}. "
    r"\textit{Speedup}: $\text{FM}_{\text{pixel}}$ wall-clock\,/\,WFM wall-clock; "
    r"${>}1$ means WFM is faster. "
    r"\textit{Mem ratio}: $\text{FM}_{\text{pixel}}$ peak GPU mem\,/\,WFM peak GPU mem; "
    r"${>}1$ means WFM uses less memory. "
    r"Dashes (---) indicate metrics not applicable to the FM baseline "
    r"(ratio columns) or missing runs. "
    r"All runs: NFE\,=\,50, 4\,GPUs, 8 ensemble members."
)

abl_lines = [
    r"\begin{table}[ht]",
    r"\caption{" + caption_abl + "}",
    r"\label{tab:inference_perf_ablation}",
    r"\resizebox{\textwidth}{!}{%",
    r"\begin{tabular}{ll l rrrr rr}",
    r"\toprule",
    (
        r"\textbf{PDE} & \textbf{Model} & \textbf{Wavelet}"
        r" & \textbf{Wall Clock (s)}"
        r" & \textbf{sec / step}"
        r" & \textbf{GPU Mem (GiB)}"
        r" & \textbf{Throughput (Mfr/s)}"
        r" & \textbf{Speedup $\times$}"
        r" & \textbf{Mem ratio $\times$} \\"
    ),
]

for pde_label in PDE_ORDER:
    abl_lines.append(r"\midrule")
    pde_tex = PDE_LATEX.get(pde_label, pde_label)
    abl_lines.append(rf"\multirow{{{N_BLOCK}}}{{*}}{{{pde_tex}}}")

    # FM row
    row_data = abl_pivot_fmt.loc[(pde_label, r"$\text{FM}_{\text{pixel}}$", "")]
    vals = " & ".join(
        str(row_data[m]).replace("×", r"$\times$").replace("—", r"\textemdash")
        for m in METRIC_COLS
    )
    abl_lines.append(rf"  & $\text{{FM}}_{{\text{{pixel}}}}$ & --- & {vals} \\")

    # WFM rows — \multirow inlined into first wavelet row
    for s_idx, scale_key in enumerate(SCALE_ORDER):
        if s_idx > 0:
            abl_lines.append(r"  \cmidrule(l){2-9}")
        scale_label = SCALE_LABELS[scale_key]
        n_w = len(WAVELET_ORDER)

        for w_idx, wavelet in enumerate(WAVELET_ORDER):
            row_data = abl_pivot_fmt.loc[(pde_label, scale_label, wavelet)]
            vals = " & ".join(
                str(row_data[m]).replace("×", r"$\times$").replace("—", r"\textemdash")
                for m in METRIC_COLS
            )
            pde_cfg  = PDE_BEST_CONFIG.get(
                next(k for k, v in PDE_LABELS.items() if v == pde_label), {}
            )
            best_name = pde_cfg.get(scale_key, "")
            is_best   = best_name.endswith(f"_{wavelet}")
            wav_tex   = rf"\textbf{{{wavelet}}}" if is_best else wavelet

            # Inline \multirow into the first row of each scale block
            if w_idx == 0:
                scale_cell = rf"\multirow{{{n_w}}}{{*}}{{{scale_label}}}"
            else:
                scale_cell = ""

            abl_lines.append(rf"  & {scale_cell} & {wav_tex} & {vals} \\")

abl_lines += [
    r"\bottomrule",
    r"\end{tabular}%",
    r"}",
    r"\end{table}",
]

latex_abl = "\n".join(abl_lines)
print(latex_abl)

\begin{table}[ht]
\caption{Full inference-performance ablation across all WFM variants (scale $\times$ wavelet) relative to $\text{FM}_{\text{pixel}}$. Columns are defined as in Table~\ref{tab:inference_perf}. \textit{Speedup}: $\text{FM}_{\text{pixel}}$ wall-clock\,/\,WFM wall-clock; ${>}1$ means WFM is faster. \textit{Mem ratio}: $\text{FM}_{\text{pixel}}$ peak GPU mem\,/\,WFM peak GPU mem; ${>}1$ means WFM uses less memory. Dashes (---) indicate metrics not applicable to the FM baseline (ratio columns) or missing runs. All runs: NFE\,=\,50, 4\,GPUs, 8 ensemble members.}
\label{tab:inference_perf_ablation}
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll l rrrr rr}
\toprule
\textbf{PDE} & \textbf{Model} & \textbf{Wavelet} & \textbf{Wall Clock (s)} & \textbf{sec / step} & \textbf{GPU Mem (GiB)} & \textbf{Throughput (Mfr/s)} & \textbf{Speedup $\times$} & \textbf{Mem ratio $\times$} \\
\midrule
\multirow{9}{*}{Turbulent Radiative Layer 2D}
  & $\text{FM}_{\text{pixel}}$ & --- & 247.8 & 0.